In [1]:
import sqlite3
from datetime import datetime, timedelta, date
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io, base64

# ─── KATEGORI (hanya 2) ───────────────────────────────────────────────────────
KATEGORI_LIST = ["Akademik", "Non Akademik"]

# ─── PALET WARNA (dari Color Palette 101) ────────────────────────────────────
FAIRY_LIGHTS      = "#FFDDBD"   # krem peach — bg card, teks terang
BLUE_TOURMALIN    = "#5FBBEF"   # biru langit — aksen utama
BLUE_HEPATICA     = "#5C71DD"   # biru medium — aksen sekunder
CANDIED_BLUEBERRY = "#522196"   # ungu gelap — header bg
BANAISAGI_PURPLE  = "#CD2382"   # magenta/pink tua — danger/high
PINK_KATYDID      = "#FF62BB"   # pink cerah — aksen warm

# shorthand
BG_MAIN   = "#120820"           # latar gelap nuansa ungu
CARD_BG   = CANDIED_BLUEBERRY   # card bg: ungu gelap
CARD_TEXT = FAIRY_LIGHTS        # teks di card: krem
ACCENT    = BLUE_TOURMALIN      # border/aksen: biru langit
ACCENT2   = BLUE_HEPATICA       # aksen sekunder: biru medium
HEADER_BG = "#1a0535"           # header sangat gelap
HEADER_TXT= FAIRY_LIGHTS        # teks header: krem
TITLE_CLR = PINK_KATYDID        # warna judul
SUB_CLR   = BLUE_TOURMALIN

# Warna per kategori di chart
KATEGORI_COLORS = {
    "Akademik":     BLUE_TOURMALIN,
    "Non Akademik": PINK_KATYDID,
}

# ─── DATABASE ─────────────────────────────────────────────────────────────────
DB_NAME = "tasks.db"

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS tasks (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        estimated_minutes INTEGER,
        deadline TEXT,
        priority TEXT,
        category TEXT,
        difficulty INTEGER DEFAULT 3,
        status TEXT DEFAULT 'pending',
        created_at TEXT DEFAULT CURRENT_TIMESTAMP,
        completed_at TEXT
    )''')
    try:
        c.execute("ALTER TABLE tasks ADD COLUMN difficulty INTEGER DEFAULT 3")
    except:
        pass
    conn.commit()
    conn.close()

init_db()

# ─── RUMUS SKOR ───────────────────────────────────────────────────────────────
def hitung_skor(difficulty, deadline_str):
    difficulty_score = (difficulty / 5) * 50
    if deadline_str:
        try:
            dl = datetime.strptime(deadline_str, "%Y-%m-%d").date()
            hari_tersisa = (dl - date.today()).days
            if hari_tersisa <= 0:   urgency_score = 50
            elif hari_tersisa <= 3: urgency_score = 40
            elif hari_tersisa <= 7: urgency_score = 25
            elif hari_tersisa <= 14:urgency_score = 15
            else:                   urgency_score = 5
        except:
            urgency_score = 0
    else:
        urgency_score = 0
    skor = round(difficulty_score + urgency_score)
    priority = "High" if skor >= 65 else "Medium" if skor >= 35 else "Low"
    return skor, priority

# ─── FUNGSI DB ────────────────────────────────────────────────────────────────
def add_task(name, minutes, deadline, category, difficulty):
    skor, priority = hitung_skor(difficulty, deadline)
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''INSERT INTO tasks (name, estimated_minutes, deadline, priority, category, difficulty, status)
                 VALUES (?,?,?,?,?,?,'pending')''',
              (name, minutes, deadline, priority, category, difficulty))
    conn.commit()
    conn.close()

def get_all_tasks_df():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query(
        "SELECT id, name, estimated_minutes, deadline, priority, category, difficulty, status FROM tasks", conn)
    conn.close()
    return df

def clear_all_tasks():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM tasks")
    conn.commit()
    conn.close()

def complete_task_by_id(task_id):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        c.execute("SELECT status FROM tasks WHERE id=?", (task_id,))
        row = c.fetchone()
        if row is None:       return False, "ID tidak ditemukan"
        if row[0] != 'pending': return False, "Tugas sudah selesai"
        now = datetime.now().isoformat()
        c.execute("UPDATE tasks SET status='completed', completed_at=? WHERE id=? AND status='pending'",
                  (now, task_id))
        conn.commit()
        return c.rowcount > 0, "Berhasil"
    except Exception as e:
        return False, str(e)
    finally:
        conn.close()

def recalc_all_scores():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT id, difficulty, deadline FROM tasks WHERE status='pending'")
    for tid, diff, dl in c.fetchall():
        skor, prio = hitung_skor(diff or 3, dl)
        c.execute("UPDATE tasks SET priority=? WHERE id=?", (prio, tid))
    conn.commit()
    conn.close()

# ─── DATA CONTOH ──────────────────────────────────────────────────────────────
def insert_sample_tasks():
    if get_all_tasks_df().empty:
        sample = [
            ("Belajar UAP",      100, "2026-06-12", "Akademik",     4),
            ("Menulis Ilmiah",    60, "2026-06-10", "Akademik",     3),
            ("Video Resume FPO",  90, "2026-06-13", "Non Akademik", 3),
            ("Notulensi",         30, "2026-06-09", "Non Akademik", 2),
            ("Presentasi Akhir",  75, "2026-06-14", "Akademik",     5),
        ]
        for nama, menit, deadline, kat, diff in sample:
            add_task(nama, menit, deadline, kat, diff)
        print("✅ Data contoh ditambahkan.\n")

insert_sample_tasks()
recalc_all_scores()

# ─── CSS ──────────────────────────────────────────────────────────────────────
def get_css():
    return f"""
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@400;700&family=Poppins:wght@300;400;600&display=swap');

        .cs-container {{
            font-family: 'Poppins', sans-serif;
            background: linear-gradient(135deg, {BG_MAIN} 0%, #1e0a3c 60%, #2a0a2e 100%);
            color: {FAIRY_LIGHTS};
            padding: 28px;
            border-radius: 24px;
            box-shadow: 0 0 40px rgba(205,35,130,0.25), 0 0 80px rgba(82,33,150,0.2);
            border: 1.5px solid {BLUE_HEPATICA};
        }}
        .cs-title {{
            font-family: 'Playfair Display', serif;
            font-size: 5em;
            font-weight: 800;
            text-align: center;
            background: linear-gradient(90deg, {BLUE_TOURMALIN}, {PINK_KATYDID}, {BLUE_HEPATICA});
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            background-clip: text;
            text-shadow: none;
            letter-spacing: 3px;
            margin-bottom: 8px;
            filter: drop-shadow(0 0 18px {BANAISAGI_PURPLE});
        }}
        .cs-sub {{
            text-align: center;
            font-style: italic;
            color: {BLUE_TOURMALIN};
            margin-bottom: 20px;
            border-top: 1px solid {BLUE_HEPATICA};
            border-bottom: 1px solid {BLUE_HEPATICA};
            display: inline-block;
            padding: 5px 24px;
            letter-spacing: 2px;
            font-size: 0.85em;
        }}
        .cs-card {{
            background: linear-gradient(135deg, rgba(82,33,150,0.55) 0%, rgba(26,5,53,0.85) 100%);
            border-radius: 18px;
            padding: 18px 20px;
            margin: 14px 0;
            border: 1px solid {BLUE_HEPATICA};
            box-shadow: 0 4px 24px rgba(205,35,130,0.12), inset 0 1px 0 rgba(255,221,189,0.08);
            color: {FAIRY_LIGHTS};
            backdrop-filter: blur(4px);
        }}
        .cs-card h3 {{
            color: {PINK_KATYDID};
            font-weight: 700;
            margin-top: 0;
            letter-spacing: 1px;
            text-transform: uppercase;
            font-size: 0.95em;
        }}
        .cs-button {{
            background: linear-gradient(90deg, {CANDIED_BLUEBERRY}, {BANAISAGI_PURPLE});
            color: {FAIRY_LIGHTS};
            border: 1px solid {PINK_KATYDID};
            border-radius: 30px;
            padding: 9px 22px;
            font-weight: 700;
            font-size: 13px;
            letter-spacing: 0.5px;
            cursor: pointer;
            transition: all 0.25s;
            box-shadow: 0 0 12px rgba(255,98,187,0.3);
        }}
        .cs-button:hover {{
            background: linear-gradient(90deg, {BANAISAGI_PURPLE}, {PINK_KATYDID});
            box-shadow: 0 0 20px rgba(255,98,187,0.6);
            transform: scale(1.03);
        }}
        .cs-table {{
            width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            background: rgba(18,8,32,0.8);
            color: {FAIRY_LIGHTS};
        }}
        .cs-table th {{
            background: linear-gradient(90deg, {CANDIED_BLUEBERRY}, #3a1070);
            color: {FAIRY_LIGHTS};
            padding: 12px 10px;
            font-weight: 600;
            border-bottom: 2px solid {PINK_KATYDID};
            white-space: nowrap;
            font-size: 12px;
            letter-spacing: 0.5px;
        }}
        .cs-table td {{
            padding: 10px;
            border-bottom: 1px solid rgba(92,113,221,0.25);
        }}
        .cs-progress {{
            border: 1px solid {BLUE_HEPATICA};
            border-radius: 30px;
            background: rgba(18,8,32,0.8);
            overflow: hidden;
        }}
        .cs-progress-bar {{
            background: linear-gradient(90deg, {BANAISAGI_PURPLE}, {PINK_KATYDID}, {BLUE_TOURMALIN}) !important;
            height: 30px;
            border-radius: 30px;
            text-align: center;
            line-height: 30px;
            color: white;
            font-weight: bold;
            transition: width 0.6s;
            text-shadow: 0 1px 3px rgba(0,0,0,0.5);
        }}
        /* Priority badges */
        .badge-high   {{ background: {BANAISAGI_PURPLE}; color: white; padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight:700; box-shadow:0 0 8px rgba(205,35,130,0.5); }}
        .badge-medium {{ background: {BLUE_HEPATICA};    color: white; padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight:700; }}
        .badge-low    {{ background: {CANDIED_BLUEBERRY};color: {FAIRY_LIGHTS}; padding: 3px 10px; border-radius: 20px; font-size: 12px; font-weight:700; }}
        /* Kategori badges */
        .badge-akademik    {{ background: {BLUE_TOURMALIN};  color: #120820; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight:700; }}
        .badge-nonakademik {{ background: {PINK_KATYDID};    color: #120820; padding: 3px 10px; border-radius: 20px; font-size: 11px; font-weight:700; }}
        /* Score bar */
        .score-bar-wrap {{ background: rgba(255,255,255,0.1); border-radius: 10px; width:80px; height:12px; display:inline-block; vertical-align:middle; margin-left:5px; }}
        .score-bar-fill {{ height:12px; border-radius:10px; }}
        /* Rank badges */
        .rank-badge {{ display:inline-block; width:26px; height:26px; border-radius:50%; text-align:center; line-height:26px; font-weight:bold; font-size:13px; }}
        .rank-1 {{ background: linear-gradient(135deg,#FFD700,#FFA500); color:#120820; box-shadow:0 0 8px #FFD700; }}
        .rank-2 {{ background: linear-gradient(135deg,#C0C0C0,#888);    color:#120820; }}
        .rank-3 {{ background: linear-gradient(135deg,#CD7F32,#8B4513); color:white; }}
        .rank-other {{ background: {CANDIED_BLUEBERRY}; color: {FAIRY_LIGHTS}; }}
        /* Formula box */
        .formula-box {{
            background: rgba(18,8,32,0.9);
            border: 1px solid {BLUE_HEPATICA};
            border-radius: 14px;
            padding: 16px 20px;
            font-family: monospace;
            font-size: 13px;
            color: {FAIRY_LIGHTS};
            margin: 8px 0;
            line-height: 2;
        }}
        .star-gold {{ color: #FFD700; letter-spacing: 2px; font-size:15px; }}
        .blink-text {{
            animation: blinkPink 0.8s infinite alternate;
            font-weight: bold;
        }}
        @keyframes blinkPink {{
            0%   {{ color: {FAIRY_LIGHTS}; }}
            100% {{ color: {PINK_KATYDID}; text-shadow: 0 0 10px {PINK_KATYDID}; }}
        }}
        @keyframes bounce {{
            from {{ transform: translateY(0px); }}
            to   {{ transform: translateY(-15px); }}
        }}
        /* Widget label overrides inside cs-card */
        .cs-card label, .cs-card .widget-label {{ color: {FAIRY_LIGHTS} !important; }}
        .cs-card .widget-text input,
        .cs-card .widget-dropdown select,
        .cs-card .widget-inttext input {{
            background: rgba(82,33,150,0.4) !important;
            color: {FAIRY_LIGHTS} !important;
            border: 1px solid {BLUE_HEPATICA} !important;
            border-radius: 8px !important;
        }}
        .cs-card .widget-slider .slider .track-highlight {{ background: {PINK_KATYDID} !important; }}
    </style>
    """

# ─── HELPER ───────────────────────────────────────────────────────────────────
def bintang(n):
    return "★" * n + "☆" * (5 - n)

def score_color(skor):
    if skor >= 65: return BANAISAGI_PURPLE
    if skor >= 35: return BLUE_HEPATICA
    return BLUE_TOURMALIN

def norm_kat(k):
    if not k: return "Non Akademik"
    kl = k.strip().lower()
    if kl in ("non akademik","non-akademik","nonakademik"): return "Non Akademik"
    if kl == "akademik": return "Akademik"
    # mapping lama → baru
    if kl in ("pribadi","olahraga","organisasi"): return "Non Akademik"
    return "Non Akademik"

# ─── TABEL ────────────────────────────────────────────────────────────────────
def tampilkan_tabel():
    df = get_all_tasks_df()
    if df.empty:
        return HTML(f"<p class='blink-text'>📭 Belum ada tugas. Silakan tambah!</p>")

    df['skor']     = df.apply(lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[0], axis=1)
    df['priority'] = df.apply(lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[1], axis=1)
    df['kat_norm'] = df['category'].apply(norm_kat)

    pending_df   = df[df['status']=='pending'].copy().sort_values('skor', ascending=False).reset_index(drop=True)
    pending_df['rank'] = range(1, len(pending_df)+1)
    completed_df = df[df['status']=='completed'].copy()
    completed_df['rank'] = None
    df_sorted = pd.concat([pending_df, completed_df])

    html = "<table class='cs-table'>"
    html += "<tr>" + "".join(f"<th>{c}</th>" for c in
            ['Rank','ID','Nama Tugas','Menit','Deadline','Kesulitan','Skor','Prioritas','Kategori','Status']) + "</tr>"

    for _, row in df_sorted.iterrows():
        is_done = row['status'] == 'completed'
        skor  = row['skor']
        diff  = int(row['difficulty']) if row['difficulty'] else 3
        prio  = row['priority']
        kat   = row['kat_norm']

        # row bg
        if is_done:
            bg = "background:rgba(82,33,150,0.15); opacity:0.55"
        elif skor >= 65:
            bg = f"background:rgba(205,35,130,0.15)"
        elif skor >= 35:
            bg = f"background:rgba(92,113,221,0.15)"
        else:
            bg = f"background:rgba(95,187,239,0.1)"

        # rank badge
        if is_done or pd.isna(row['rank']):
            rank_html = "<td style='text-align:center'>—</td>"
        else:
            r = int(row['rank'])
            cls = {1:'rank-1',2:'rank-2',3:'rank-3'}.get(r,'rank-other')
            rank_html = f"<td style='text-align:center'><span class='rank-badge {cls}'>{r}</span></td>"

        prio_html = (f"<span class='badge-high'>🔴 High</span>" if prio=='High'
                else f"<span class='badge-medium'>🟡 Medium</span>" if prio=='Medium'
                else f"<span class='badge-low'>🟢 Low</span>")

        kat_cls = "badge-akademik" if kat=="Akademik" else "badge-nonakademik"
        kat_html = f"<span class='{kat_cls}'>{kat}</span>"

        sc = score_color(skor)
        skor_html = (f"<span style='font-weight:bold;color:{sc}'>{skor}</span>"
                     f"<span class='score-bar-wrap'>"
                     f"<span class='score-bar-fill' style='width:{skor}%;background:{sc}'></span></span>")

        diff_html = f"<span class='star-gold'>{bintang(diff)}</span>"
        status_html = '✅ Selesai' if is_done else '⏳ Pending'

        html += f"<tr style='{bg}'>"
        html += rank_html
        html += f"<td>{int(row['id'])}</td>"
        html += f"<td><b>{row['name']}</b></td>"
        html += f"<td>{row['estimated_minutes']}</td>"
        html += f"<td>{row['deadline'] or '—'}</td>"
        html += f"<td>{diff_html}</td>"
        html += f"<td>{skor_html}</td>"
        html += f"<td>{prio_html}</td>"
        html += f"<td>{kat_html}</td>"
        html += f"<td>{status_html}</td>"
        html += "</tr>"
    html += "</table>"
    return HTML(get_css() + html)

# ─── FORMULA ──────────────────────────────────────────────────────────────────
def tampil_formula():
    html = f"""
    <div class='formula-box'>
        <b style='color:{PINK_KATYDID};font-size:15px'>⚙️ Rumus Penghitungan Skor Prioritas</b><br><br>
        <span style='color:{BLUE_TOURMALIN}'>Skor (0–100) = Skor Kesulitan + Skor Urgensi Deadline</span><br><br>
        <b style='color:{FAIRY_LIGHTS}'>Skor Kesulitan</b> = (tingkat_kesulitan / 5) × 50 &nbsp;→ maks 50 poin<br>
        &nbsp;&nbsp;⭐ ×1→+10 &nbsp;⭐×2→+20 &nbsp;⭐×3→+30 &nbsp;⭐×4→+40 &nbsp;⭐×5→+50<br><br>
        <b style='color:{FAIRY_LIGHTS}'>Skor Urgensi Deadline</b>:<br>
        &nbsp;&nbsp;<span style='color:{PINK_KATYDID}'>● Sudah lewat / hari ini → +50</span><br>
        &nbsp;&nbsp;<span style='color:{BANAISAGI_PURPLE}'>● 1–3 hari → +40</span> &nbsp;
              <span style='color:{BLUE_HEPATICA}'>● 4–7 hari → +25</span> &nbsp;
              <span style='color:{BLUE_TOURMALIN}'>● 8–14 hari → +15</span> &nbsp;
              <span style='color:{FAIRY_LIGHTS}'>● &gt;14 hari → +5</span><br><br>
        <b>Klasifikasi:</b> &nbsp;
        <span class='badge-high'>🔴 High ≥65</span> &nbsp;
        <span class='badge-medium'>🟡 Medium 35–64</span> &nbsp;
        <span class='badge-low'>🟢 Low &lt;35</span>
    </div>
    """
    return HTML(get_css() + html)

# ─── PROGRESS ─────────────────────────────────────────────────────────────────
def tampil_progress_html():
    df = get_all_tasks_df()
    if df.empty:
        return HTML("<p>Belum ada tugas.</p>")
    total   = len(df)
    selesai = len(df[df['status']=='completed'])
    pending = total - selesai
    percent = (selesai / total) * 100 if total > 0 else 0
    html = f"""
    <div class='cs-progress'>
        <div class='cs-progress-bar' style='width:{percent}%;'>
            {percent:.1f}%
        </div>
    </div>
    <p style='margin-top:10px;color:{BLUE_TOURMALIN}'>✨ Selesai: <b>{selesai}</b> &nbsp;|&nbsp; Pending: <b>{pending}</b> &nbsp;|&nbsp; Total: <b>{total}</b></p>
    """
    if percent == 100:
        html += f"""
        <div style="text-align:center;font-size:40px;animation:bounce 0.5s infinite alternate;">😊🌸🎉✨🥳</div>
        <div class='blink-text' style='text-align:center;font-size:22px;'>🌟 SELAMAT! SEMUA TUGAS SELESAI! 🌟</div>
        <script>
        (function(){{
            if(typeof canvasConfetti==='undefined'){{
                var s=document.createElement('script');
                s.src='https://cdn.jsdelivr.net/npm/canvas-confetti@1';
                s.onload=function(){{canvasConfetti({{particleCount:300,spread:100,origin:{{y:0.6}},colors:['{PINK_KATYDID}','{BLUE_TOURMALIN}','{FAIRY_LIGHTS}']}});}};
                document.head.appendChild(s);
            }} else {{
                canvasConfetti({{particleCount:300,spread:100,origin:{{y:0.6}},colors:['{PINK_KATYDID}','{BLUE_TOURMALIN}','{FAIRY_LIGHTS}']}});
            }}
        }})();
        </script>
        """
    return HTML(get_css() + html)

# ─── JADWAL ───────────────────────────────────────────────────────────────────
def buat_jadwal():
    df = get_all_tasks_df()
    pending = df[df['status']=='pending'].copy()
    if pending.empty:
        return HTML(f"<p style='color:{BLUE_TOURMALIN}'>🎉 Semua tugas selesai. Tidak ada jadwal hari ini.</p>")
    pending['skor'] = pending.apply(
        lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[0], axis=1)
    pending = pending.sort_values('skor', ascending=False)

    total_menit = 8*60
    used   = 0
    mulai  = datetime.strptime("09:00", "%H:%M")
    html   = f"<h3 style='color:{PINK_KATYDID}'>📅 Jadwal Hari Ini (8 jam, mulai 09:00) — diurutkan by Skor</h3>"
    html  += "<ul style='list-style:none;padding-left:0'>"
    for _, row in pending.iterrows():
        dur  = row['estimated_minutes']
        skor = row['skor']
        diff = int(row['difficulty']) if row['difficulty'] else 3
        kat  = norm_kat(row['category'])
        warna = BANAISAGI_PURPLE if skor>=65 else BLUE_HEPATICA if skor>=35 else CANDIED_BLUEBERRY
        border = PINK_KATYDID if skor>=65 else BLUE_TOURMALIN
        if used + dur <= total_menit:
            selesai_t = mulai + timedelta(minutes=dur)
            kat_badge = f"<span style='background:{'"+BLUE_TOURMALIN+"' if kat=='Akademik' else '"+PINK_KATYDID+"'};color:#120820;border-radius:12px;padding:1px 8px;font-size:11px;font-weight:bold'>{kat}</span>"
            html += (f"<li style='background:{warna};color:white;margin:8px;padding:11px 14px;"
                     f"border-radius:14px;border-left:5px solid {border}'>"
                     f"<b>{mulai.strftime('%H:%M')} – {selesai_t.strftime('%H:%M')}</b>"
                     f" &nbsp;|&nbsp; {row['name']} ({dur} mnt)"
                     f" &nbsp;|&nbsp; Skor: <b>{skor}</b>"
                     f" &nbsp;|&nbsp; <span style='color:#FFD700'>{bintang(diff)}</span>"
                     f" &nbsp;{kat_badge}</li>")
            mulai  = selesai_t
            used  += dur
        else:
            html += (f"<li style='background:rgba(255,255,255,0.07);color:#aaa;margin:8px;padding:10px;"
                     f"border-radius:14px'>⚠️ {row['name']} tidak muat hari ini → jadwalkan besok</li>")
    html += f"</ul><p style='color:{BLUE_TOURMALIN}'>⏱ Total terpakai: <b>{used/60:.1f} jam</b> dari 8 jam</p>"
    return HTML(get_css() + html)

# ─── PLOT KATEGORI ────────────────────────────────────────────────────────────
PLOT_BG   = "#120820"
PLOT_GRID = "#2a1050"

def fig_to_base64(fig):
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', facecolor=PLOT_BG, dpi=110)
    buf.seek(0)
    b64 = base64.b64encode(buf.read()).decode()
    plt.close(fig)
    return b64

def buat_plot_kategori():
    df = get_all_tasks_df()
    if df.empty:
        return HTML(f"<p style='color:{BLUE_TOURMALIN}'>Belum ada data.</p>")

    df['skor']     = df.apply(lambda r: hitung_skor(r['difficulty'] or 3, r['deadline'])[0], axis=1)
    df['kat_norm'] = df['category'].apply(norm_kat)
    colors_map     = KATEGORI_COLORS.copy()

    # ── PIE: distribusi jumlah tugas ─────────────────────────────────────────
    kat_count  = df.groupby('kat_norm').size().reindex(KATEGORI_LIST, fill_value=0)
    kat_count  = kat_count[kat_count > 0]
    fig1, ax1  = plt.subplots(figsize=(4.8, 4.4), facecolor=PLOT_BG)
    ax1.set_facecolor(PLOT_BG)
    pie_colors = [colors_map.get(k, "#888") for k in kat_count.index]
    wedges, texts, autotexts = ax1.pie(
        kat_count.values, labels=kat_count.index,
        autopct='%1.0f%%', startangle=140, colors=pie_colors,
        pctdistance=0.72, wedgeprops=dict(linewidth=2.5, edgecolor=PLOT_BG),
        textprops={'color': FAIRY_LIGHTS, 'fontsize': 10}
    )
    for at in autotexts:
        at.set_color('white'); at.set_fontsize(11); at.set_fontweight('bold')
    ax1.set_title('Distribusi Tugas\nper Kategori', color=PINK_KATYDID,
                  fontsize=12, fontweight='bold', pad=12)
    b64_1 = fig_to_base64(fig1)

    # ── BAR H: rata-rata skor ─────────────────────────────────────────────────
    skor_by_kat = df.groupby('kat_norm')['skor'].mean().reindex(KATEGORI_LIST).dropna()
    fig2, ax2   = plt.subplots(figsize=(4.8, 4.4), facecolor=PLOT_BG)
    ax2.set_facecolor(PLOT_BG)
    bar_colors  = [colors_map.get(k, "#888") for k in skor_by_kat.index]
    bars = ax2.barh(skor_by_kat.index, skor_by_kat.values,
                    color=bar_colors, edgecolor=PLOT_BG, linewidth=1.5, height=0.45)
    for bar, val in zip(bars, skor_by_kat.values):
        ax2.text(val + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}', va='center', color=FAIRY_LIGHTS, fontsize=11, fontweight='bold')
    ax2.set_xlim(0, 108)
    ax2.set_title('Rata-rata Skor Prioritas\nper Kategori', color=PINK_KATYDID,
                  fontsize=12, fontweight='bold', pad=12)
    ax2.tick_params(colors=FAIRY_LIGHTS, labelsize=10)
    for spine in ax2.spines.values(): spine.set_color(PLOT_GRID)
    ax2.set_xlabel('Skor (0–100)', color=BLUE_TOURMALIN, fontsize=9)
    ax2.axvline(65, color=BANAISAGI_PURPLE, linestyle='--', linewidth=1.2, alpha=0.8, label='High ≥65')
    ax2.axvline(35, color=BLUE_HEPATICA,   linestyle='--', linewidth=1.2, alpha=0.8, label='Medium ≥35')
    ax2.legend(fontsize=8, facecolor=PLOT_BG, labelcolor=FAIRY_LIGHTS, edgecolor=PLOT_GRID)
    b64_2 = fig_to_base64(fig2)

    # ── STACKED BAR: pending vs selesai ──────────────────────────────────────
    kat_status = df.groupby(['kat_norm','status']).size().unstack(fill_value=0)
    if 'pending'   not in kat_status.columns: kat_status['pending']   = 0
    if 'completed' not in kat_status.columns: kat_status['completed'] = 0
    kat_status = kat_status.reindex(KATEGORI_LIST).fillna(0)

    fig3, ax3  = plt.subplots(figsize=(4.8, 4.4), facecolor=PLOT_BG)
    ax3.set_facecolor(PLOT_BG)
    x          = range(len(kat_status.index))
    b_colors   = [colors_map.get(k,"#888") for k in kat_status.index]
    ax3.bar(x, kat_status['pending'],   width=0.45, color=b_colors,    alpha=0.92, label=' Pending',  edgecolor=PLOT_BG)
    ax3.bar(x, kat_status['completed'], width=0.45, bottom=kat_status['pending'],
            color='#27ae60', alpha=0.75, label=' Selesai', edgecolor=PLOT_BG)
    ax3.set_xticks(list(x))
    ax3.set_xticklabels(kat_status.index, color=FAIRY_LIGHTS, fontsize=10)
    ax3.set_title('Pending vs Selesai\nper Kategori', color=PINK_KATYDID,
                  fontsize=12, fontweight='bold', pad=12)
    ax3.tick_params(colors=FAIRY_LIGHTS, labelsize=10)
    for spine in ax3.spines.values(): spine.set_color(PLOT_GRID)
    ax3.set_ylabel('Jumlah Tugas', color=BLUE_TOURMALIN, fontsize=9)
    ax3.legend(fontsize=9, facecolor=PLOT_BG, labelcolor=FAIRY_LIGHTS, edgecolor=PLOT_GRID)
    b64_3 = fig_to_base64(fig3)

    html = f"""
    <div style='display:flex;flex-wrap:wrap;gap:14px;justify-content:center;margin-top:8px;'>
        <div style='background:rgba(82,33,150,0.3);border:1px solid {BLUE_HEPATICA};border-radius:14px;padding:12px;text-align:center;'>
            <img src='data:image/png;base64,{b64_1}' style='max-width:270px;border-radius:8px'/>
        </div>
        <div style='background:rgba(82,33,150,0.3);border:1px solid {BLUE_HEPATICA};border-radius:14px;padding:12px;text-align:center;'>
            <img src='data:image/png;base64,{b64_2}' style='max-width:270px;border-radius:8px'/>
        </div>
        <div style='background:rgba(82,33,150,0.3);border:1px solid {BLUE_HEPATICA};border-radius:14px;padding:12px;text-align:center;'>
            <img src='data:image/png;base64,{b64_3}' style='max-width:270px;border-radius:8px'/>
        </div>
    </div>
    """
    return HTML(get_css() + html)

# ─── WIDGETS ──────────────────────────────────────────────────────────────────
nama_input = widgets.Text(
    placeholder="Nama tugas", description="📝 Nama:",
    layout=widgets.Layout(width='500px'))
waktu_input = widgets.IntSlider(
    min=5, max=500, step=5, value=30, description="⏱ Menit:",
    layout=widgets.Layout(width='500px'))
deadline_input = widgets.DatePicker(
    description="📅 Deadline:", layout=widgets.Layout(width='500px'))
kategori_input = widgets.Dropdown(
    options=KATEGORI_LIST, value="Akademik",
    description="📂 Kategori:", layout=widgets.Layout(width='500px'))
difficulty_input = widgets.RadioButtons(
    options=[(f'{bintang(i)}  ({i})', i) for i in range(1, 6)],
    value=3, description="🎯 Kesulitan:",
    layout=widgets.Layout(width='500px'))

out_score_preview = widgets.Output()

def update_score_preview(*args):
    dl   = deadline_input.value.isoformat() if deadline_input.value else None
    skor, prio = hitung_skor(difficulty_input.value, dl)
    sc   = score_color(skor)
    pc   = {  "High": BANAISAGI_PURPLE,
              "Medium": BLUE_HEPATICA,
              "Low": BLUE_TOURMALIN}[prio]
    with out_score_preview:
        clear_output(wait=True)
        display(HTML(get_css() + f"""
        <div style='background:rgba(18,8,32,0.9);border:1px solid {BLUE_HEPATICA};border-radius:12px;
                    padding:12px 18px;display:inline-block;margin-top:6px;'>
            <span style='color:{FAIRY_LIGHTS};font-size:13px;opacity:0.7'>Estimasi Skor: </span>
            <span style='font-size:24px;font-weight:bold;color:{sc}'>{skor}</span>
            <span class='score-bar-wrap' style='width:120px;vertical-align:middle;margin:0 10px'>
                <span class='score-bar-fill' style='width:{skor}%;background:{sc}'></span>
            </span>
            <span style='color:{pc};font-weight:bold;font-size:14px'>→ {prio}</span>
        </div>"""))

difficulty_input.observe(update_score_preview, names='value')
deadline_input.observe(update_score_preview,   names='value')

btn_tambah = widgets.Button(description="➕ TAMBAH TUGAS", layout=widgets.Layout(width='300px'))
btn_reset  = widgets.Button(description="🗑 HAPUS SEMUA",  layout=widgets.Layout(width='300px'))
btn_tambah.add_class('cs-button')
btn_reset.add_class('cs-button')

out_tabel    = widgets.Output()
out_progress = widgets.Output()
out_jadwal   = widgets.Output()
out_status   = widgets.Output()
out_formula  = widgets.Output()
out_plot_kat = widgets.Output()

def refresh_all():
    recalc_all_scores()
    with out_tabel:
        clear_output(wait=True); display(HTML(get_css())); display(tampilkan_tabel())
    with out_progress:
        clear_output(wait=True); display(HTML(get_css())); display(tampil_progress_html())
    with out_jadwal:
        clear_output(wait=True); display(HTML(get_css())); display(buat_jadwal())
    with out_formula:
        clear_output(wait=True); display(tampil_formula())
    with out_plot_kat:
        clear_output(wait=True); display(HTML(get_css())); display(buat_plot_kategori())

def tambah(btn):
    with out_status:
        clear_output()
        nama = nama_input.value.strip()
        if not nama:
            print("⚠️ Nama tugas harus diisi!"); return
        dl = deadline_input.value.isoformat() if deadline_input.value else None
        add_task(nama, waktu_input.value, dl, kategori_input.value, difficulty_input.value)
        skor, prio = hitung_skor(difficulty_input.value, dl)
        print(f"✅ '{nama}' ditambahkan! Skor: {skor} → Prioritas: {prio}")
        nama_input.value = ""; waktu_input.value = 30
        deadline_input.value = None; kategori_input.value = "Akademik"
        difficulty_input.value = 3
        refresh_all(); update_score_preview()

def hapus_semua(btn):
    with out_status:
        clear_output()
        clear_all_tasks()
        print("🗑 Semua tugas telah dihapus.")
        refresh_all()
        insert_sample_tasks()
        refresh_all()

btn_tambah.on_click(tambah)
btn_reset.on_click(hapus_semua)

id_selesai  = widgets.IntText(description="ID Tugas:", value=1, layout=widgets.Layout(width='150px'))
btn_selesai = widgets.Button(description="✔️ TANDAI SELESAI", layout=widgets.Layout(width='180px'))
btn_selesai.add_class('cs-button')

def selesaikan(btn):
    with out_status:
        clear_output()
        tid = id_selesai.value
        sukses, pesan = complete_task_by_id(tid)
        if sukses:
            print(f"✅ Tugas ID {tid} telah diselesaikan!"); refresh_all()
        else:
            print(f"❌ Gagal: {pesan}. Cek ID {tid} dan pastikan status masih Pending.")
        id_selesai.value = 1

btn_selesai.on_click(selesaikan)

# ─── TAMPILAN UTAMA ───────────────────────────────────────────────────────────
display(HTML(get_css()))
display(HTML(f"""
<div class='cs-container'>
    <div class='cs-title'>ChronoScore</div>
    <div style='text-align:center'><span class='cs-sub'>TO DO LIST BY KELOMPOK 6</span></div>
"""))

display(HTML(f"<div class='cs-card'><h3>⚙️ SISTEM SCORING PRIORITAS</h3>"))
display(out_formula)
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3 style='text-align:center'>✍️ TAMBAH TUGAS BARU</h3>"
             "<div style='display:flex;justify-content:center;'>"))
display(widgets.VBox([
    nama_input, waktu_input, deadline_input, kategori_input,
    difficulty_input, out_score_preview,
    widgets.HBox([btn_tambah, btn_reset])
], layout=widgets.Layout(align_items='center')))
display(HTML("</div></div>"))

display(HTML(f"<div class='cs-card'><h3>📋 DAFTAR TUGAS (diurutkan by Skor)</h3>"))
display(out_tabel)
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3>✅ TANDAI TUGAS SELESAI</h3>"))
display(widgets.HBox([id_selesai, btn_selesai]))
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3>📊 PROGRESS & JADWAL</h3>"))
display(out_progress)
display(out_jadwal)
display(HTML("</div>"))

display(HTML(f"<div class='cs-card'><h3>📈 ANALISIS PER KATEGORI</h3>"))
display(out_plot_kat)
display(HTML("</div>"))

display(HTML("</div>"))
display(out_status)

refresh_all()
update_score_preview()

Output()

Output()

Output()

Output()

Output()

Output()